# DoorDash Delivery ETA — Etapa 08: Interpretabilidade e Investigação de Erros

Este notebook investiga o modelo oficial congelado (`models/final_hgb_eta_pipeline.joblib`) **sem modificá-lo, retreiná-lo ou usar o conjunto de teste**:
1. Auditoria do pipeline salvo e das 21 features confirmadas.
2. Importância global via *Permutation Importance* (`neg_mean_absolute_error`) e análise de dependência parcial na validação (`n = 28.808`).
3. Investigação de erros por faixa, mercado, horário e perfil dos pedidos.
4. Investigação de atrasos extremos (`> 120 min`) diferenciando anomalias estatísticas, possíveis falhas de registro, atrasos reais e limitações do modelo.

In [ ]:
import sys
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.inspection import permutation_importance, partial_dependence

sns.set_theme(style="whitegrid")
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preparation import prepare_datasets
from src.feature_engineering import build_features
from src.temporal_validation import temporal_split_with_embargo
from src.model_training import (
    prepare_feature_frame,
    evaluate_by_duration_band,
    CONFIRMED_MODEL_FEATURES,
    TARGET_COL,
)

pipe = joblib.load(PROJECT_ROOT / "models" / "final_hgb_eta_pipeline.joblib")
_, _, df_model, _ = prepare_datasets(PROJECT_ROOT / "data" / "raw" / "historical_data.csv")
df_feat = build_features(df_model)
split = temporal_split_with_embargo(df_feat)

X_val = prepare_feature_frame(split.val, CONFIRMED_MODEL_FEATURES)
y_val = split.val[TARGET_COL].values
y_val_pred = pipe.predict(X_val)

print("Features de entrada (21):", list(pipe.named_steps["prep"].feature_names_in_))
print("Colunas após One-Hot Encoding:", len(pipe.named_steps["prep"].get_feature_names_out()))

## 1. Permutation Importance e Dependência Parcial (Validação)

In [ ]:
perm = permutation_importance(
    pipe, X_val, y_val,
    scoring="neg_mean_absolute_error",
    n_repeats=10, random_state=42, n_jobs=-1,
)
imp_df = pd.DataFrame({
    "feature": CONFIRMED_MODEL_FEATURES,
    "importance_mean_mae": np.round(perm.importances_mean, 5),
    "importance_std": np.round(perm.importances_std, 5),
}).sort_values("importance_mean_mae", ascending=False).reset_index(drop=True)
imp_df["rank"] = imp_df.index + 1
display(imp_df)

fig, ax = plt.subplots(figsize=(10, 7))
imp_plot = imp_df.sort_values("importance_mean_mae", ascending=True)
ax.barh(imp_plot["feature"], imp_plot["importance_mean_mae"], xerr=imp_plot["importance_std"], color="#2b5c8f")
ax.axvline(0, color="black", linestyle="--", linewidth=0.8)
ax.set_title("Permutation Importance (Validação, MAE)")
ax.set_xlabel("Aumento médio no MAE (minutos)")
fig.tight_layout()
plt.show()

In [ ]:
top_num_feats = [f for f in imp_df["feature"] if f not in ["market_id", "store_primary_category", "order_protocol", "created_time_block"]][:4]
X_val_sample = X_val.sample(min(5000, len(X_val)), random_state=42)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feat in zip(axes.flat, top_num_feats):
    pd_res = partial_dependence(pipe, X_val_sample, features=[feat], grid_resolution=25)
    ax.plot(pd_res["grid_values"][0], pd_res["average"][0], marker="o", color="#2b5c8f", linewidth=2)
    ax.set_title(f"Dependência Parcial: {feat}")
    ax.set_xlabel(feat)
    ax.set_ylabel("Duração Prevista Média (min)")
fig.tight_layout()
plt.show()

## 2. Investigação dos Erros na Validação

In [ ]:
val_diag = split.val.copy()
val_diag["y_true"] = y_val
val_diag["y_pred"] = y_val_pred
val_diag["bias"] = y_val_pred - y_val
val_diag["abs_err"] = np.abs(val_diag["bias"])

display(evaluate_by_duration_band(split.val[TARGET_COL], y_val_pred))

p95_err = val_diag["abs_err"].quantile(0.95)
val_diag["is_high_err"] = val_diag["abs_err"] >= p95_err
print(f"Limiar Top 5% maiores erros (P95): {p95_err:.2f} min")
display(val_diag.groupby("is_high_err")[["y_true", "y_pred", "subtotal", "total_items", "average_item_price", "historical_store_order_count"]].median().round(2))

val_diag["market_str"] = val_diag["market_id"].astype("object").fillna("NA").astype(str)
display(val_diag.groupby("market_str").agg(n=("y_true", "count"), MAE=("abs_err", "mean"), MedAE=("abs_err", "median"), Mean_Bias=("bias", "mean")).round(2).sort_values("MAE", ascending=False))

## 3. Investigação de Atrasos Extremos (`> 120 min`)

In [ ]:
print(f"Treino > 120m: {(split.train[TARGET_COL] > 120).sum()} | > 24h: {(split.train[TARGET_COL] > 1440).sum()}")
print(f"Validação > 120m: {(val_diag[TARGET_COL] > 120).sum()} | > 24h: {(val_diag[TARGET_COL] > 1440).sum()}")

display(val_diag.sort_values("abs_err", ascending=False)[
    ["created_at", "actual_delivery_time", "market_id", "store_id", "store_primary_category", "total_items", "subtotal", "y_true", "y_pred", "abs_err"]
].head(15))